# UEMR HT-Step: phần bổ sung (docs/uemr/HTSTEP_BO_SUNG.md)

Val đã chấm ở lượt chính (protocol `a4ca43db…`). Đây là phân tích bổ sung; câu kết luận chính không đổi. Trước khi chạy: tải `feats.npz`, `student_r8.npz`, `student_r8_gt.npz`, `student_r8_rand.npz`, `student_r7_gt.npz` từ `D:/Học/KL/output/htstep_uemr/bo_sung/uniav/` lên `MyDrive/Colab Notebooks/code KL/Fushion/uemr_htstep_run/bo_sung_in/`. Chạy `SMOKE = True` trước, rồi `SMOKE = False`, Run all.

In [ ]:
# ===== 0. Cấu hình =====
FUSHION_REF = '93538dc58d94ff78d8d4f9db5280620e6a087fd5'   # commit trên nhánh htstep chứa configs/htstep_uemr_bo_sung_protocol.json
PROTO_SHA16 = 'b65ee046d8182742'   # sha256[:16] của protocol bổ sung tại FUSHION_REF
MAIN_SHA16  = 'a4ca43db9829842a'  # protocol lượt chính (v4), không đổi
SMOKE       = True       # True: 5 video (chỉ CHECK, shape, tốc độ; ghi bo_sung/smoke, ô 12 xoá). Xong đổi False rồi Run all
RUN_TEACHER = True       # mục 5.5 (ưu tiên 2): encode_av thầy trên đoạn R6
RUN_R6K, RUN_R7FILL, RUN_PT = False, False, False   # phần F, mặc định tắt
SEED = 0
HANLI_REF, OMNI_FIX_REF, OMNI_AV_REF = '77fc4fa', 'ab772eb', '57a7f1f'
FLAGS = {k: v for k, v in dict(globals()).items() if k.startswith('RUN_') or k in ('FUSHION_REF', 'PROTO_SHA16', 'MAIN_SHA16', 'SEED', 'SMOKE')}

In [ ]:
# ===== 1. Drive, code (Fushion, HanLi, Omni-fix), thư viện, unit test =====
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, sys, json, glob, shutil, subprocess, time, hashlib, platform, csv
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'
os.environ.pop('HF_XET_HIGH_PERFORMANCE', None)
T_START = time.time()
FUSHION, HANLI, OMNI_FIX, OMNI_AV = '/content/code/Fushion_UEMR', '/content/code/HanLi-Retriever', '/content/code/Omni-fix', '/content/code/Omni-av'
os.makedirs('/content/code', exist_ok=True)

def clone(url, path, ref, *branch):
    os.chdir('/content')
    shutil.rmtree(path, ignore_errors=True)
    subprocess.run(['git', 'clone', '-q', *branch, url, path], check=True)
    r = subprocess.run(['git', '-C', path, 'checkout', '-q', ref], capture_output=True, text=True)
    assert r.returncode == 0, f'không checkout được {ref} của {url}: {r.stderr} (đã push chưa?)'
    return subprocess.run(['git', '-C', path, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip()

COMMIT = clone('https://github.com/chien24/Fushion_UEMR.git', FUSHION, FUSHION_REF)
HANLI_COMMIT = clone('https://github.com/im-xiaoming/HanLi-Retriever.git', HANLI, HANLI_REF)
OMNI_COMMIT = clone('https://github.com/im-xiaoming/Omni-Retriever-Custom.git', OMNI_FIX, OMNI_FIX_REF, '-b', 'htstep_lora')
OMNI_AV_COMMIT = clone('https://github.com/im-xiaoming/Omni-Retriever-Custom.git', OMNI_AV, OMNI_AV_REF, '-b', 'htstep-av-extract')
print('Fushion :', COMMIT); print('HanLi   :', HANLI_COMMIT); print('Omni-fix:', OMNI_COMMIT); print('Omni-av :', OMNI_AV_COMMIT)
git = lambda *a: subprocess.run(['git', '-C', FUSHION, *a], capture_output=True, text=True).stdout.strip()
if FUSHION not in sys.path:
    sys.path.insert(0, FUSHION)
os.environ['PYTHONPATH'] = FUSHION + ':' + os.environ.get('PYTHONPATH', '')
os.chdir(FUSHION)

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr)[-3000:]
    if out.strip():
        print(out)
    if check and r.returncode:
        raise RuntimeError(f'lệnh lỗi ({r.returncode}): {cmd[:200]}')
    return r

def check(name, ok, detail=''):
    print(f"CHECK {name}: {'OK' if ok else 'FAIL'}" + (f' | {detail}' if detail else ''))
    assert ok, f'CHECK {name} FAIL: {detail}'

sh(f'pip install -q -r {FUSHION}/requirements.txt')
sh('pip uninstall -y -q torchao', check=False)
import numpy as np, pandas as pd, torch
from IPython.display import display
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có'
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0.0
ENV = {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__, 'gpu': gpu, 'gpu_GB': round(GPU_GB, 1),
       'fushion': COMMIT, 'hanli': HANLI_COMMIT, 'omni_fix': OMNI_COMMIT, 'omni_av': OMNI_AV_COMMIT, 'smoke': SMOKE}
print(json.dumps(ENV, indent=1, ensure_ascii=False))
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_uemr_htstep.py', 'tests/test_uemr_htstep_bo_sung.py', '-q', '-p', 'no:cacheprovider'],
                   cwd=FUSHION, capture_output=True, text=True, env={**os.environ, 'UNIAV_REPO': HANLI})
print(r.stdout[-1200:])
check('unit test (lượt chính + bổ sung)', r.returncode == 0 and ' skipped' not in r.stdout.splitlines()[-1], r.stdout.splitlines()[-1] if r.stdout else r.stderr[-300:])

def sha256(path, bs=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(bs), b''):
            h.update(chunk)
    return h.hexdigest()

def wj(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=str)
    os.replace(tmp, path)

def rj(path, default=None):
    return json.load(open(path, encoding='utf-8')) if os.path.exists(path) else default

DRIVE = '/content/drive/MyDrive'
ROOT  = f'{DRIVE}/Colab Notebooks/code KL/Fushion/uemr/htstep'          # chỉ đọc: cache UniAV của lượt chính
HM    = f'{DRIVE}/Colab Notebooks/code KL/Fushion/uemr_htstep_run'      # lượt chính: omni/best, omni/student_text (đọc; store clip best được ghi thêm)
BIN   = f'{HM}/bo_sung_in'                                              # user tải lên từ D:/Học/KL/output/htstep_uemr/bo_sung/uniav
HB    = f'{HM}/bo_sung' + ('/smoke' if SMOKE else '')                   # mọi đầu ra của phần bổ sung
LOCAL = '/content/smoke_bs' if SMOKE else '/content/bs'
if SMOKE:
    shutil.rmtree(HB, ignore_errors=True); shutil.rmtree(LOCAL, ignore_errors=True)
VID = '/content/videos_val'
RUN_DRIVE = f'{DRIVE}/omniretriever/htstep_lora/lora_bs32'
A = '/content/hts'
for d in (HB, VID, A, LOCAL):
    os.makedirs(d, exist_ok=True)

In [ ]:
# ===== 2. Dữ liệu, vector lượt chính, tái lập mảng hạng (mục 2.1) =====
from huggingface_hub import hf_hub_download, snapshot_download
from uemr_htstep import data as hd, metrics as hm, score as sc, bo_sung as bs, partitions as hp
from omni_retrieval.collect import local_npz
t0 = time.time()
PR = json.load(open(f'{FUSHION}/configs/htstep_uemr_protocol.json', encoding='utf-8'))
check('protocol lượt chính không đổi', sha256(f'{FUSHION}/configs/htstep_uemr_protocol.json')[:16] == MAIN_SHA16)
D = PR['data']
fetch_meta = lambda name: hf_hub_download(D['val_manifest']['hf'], name, repo_type='dataset', local_dir=A)
val_p, items_p = fetch_meta(D['val_manifest']['file']), fetch_meta(D['items_val']['file'])
check('sha256 val manifest', sha256(val_p) == D['val_manifest']['sha256'])
check('sha256 items_val', sha256(items_p) == D['items_val']['sha256'])
VAL_ROWS = hd.read_jsonl(val_p)
KHO = D['gallery']
DUR = {v: float(D['durations'][v]) for v in KHO}
CAPS = hd.queries(VAL_ROWS, KHO)
check('số câu hỏi = protocol', len(CAPS) == D['queries'], str(len(CAPS)))
TEACHER_DIR = snapshot_download('nguyenminh04/htstep-val-seen-feats', repo_type='dataset', allow_patterns=['omni_av/*.npz'], local_dir=f'{A}/val_feats') + '/omni_av'
TEACHER = hd.teacher_vectors(TEACHER_DIR)
SMOKE_VIDS = hd.smoke_videos(TEACHER, DUR, {r['video_id'] for r in VAL_ROWS if r['src_transcoded']})
VIDS = SMOKE_VIDS if SMOKE else KHO
OM = lambda tag: f'{HM}/omni/{tag}'
MAIN_RES = f'{FUSHION}/results/uemr_htstep/{MAIN_SHA16[:8]}'
MAIN_SEG_HASH = rj(f'{MAIN_RES}/segments_SOURCE.json')['seg_hash']
for name in ('best', 'student_text'):      # vector lượt chính: Drive, không có thì tải từ HF (đã đẩy ở ô 10b lượt chính)
    need = f"{OM(name)}/db/{'segments.npz' if name == 'best' else 'text.npz'}"
    if not os.path.exists(need):
        snapshot_download('GraceFeel/encode_omni', repo_type='dataset', allow_patterns=[f'htstep_uemr/{MAIN_SHA16[:8]}/{name}/*'], local_dir='/content/hf_omni')
        shutil.copytree(f'/content/hf_omni/htstep_uemr/{MAIN_SHA16[:8]}/{name}', OM(name), dirs_exist_ok=True)
meta = rj(f"{OM('best')}/db/meta.json", {})
check('vector best của lượt chính đúng bộ đoạn đã khoá', meta.get('seg_hash') == MAIN_SEG_HASH and meta.get('protocol_sha16') == MAIN_SHA16, str({k: meta.get(k) for k in ('seg_hash', 'protocol_sha16')}))
UNIAV_DIR = f"{ROOT}/{D['uniav_cache_dir']}"
for r_, s_ in D['student_sha256'].items():
    check(f'sha256 student_{r_}.npz (lượt chính)', sha256(f'{UNIAV_DIR}/student_{r_}.npz') == s_)
DB = str(local_npz(f"{OM('best')}/db", f'{LOCAL}/main_best', ('segments.npz', 'text.npz')))
ST = str(local_npz(f"{OM('student_text')}/db", f'{LOCAL}/main_student', ('text.npz',)))
BETA, NB, BSEED = float(PR['params']['beta']), PR['eval']['bootstrap']['n_boot'], PR['eval']['bootstrap']['seed']
MAIN = sc.Main(DB, ST, UNIAV_DIR, CAPS)
REP = sc.reproduce(MAIN, MAIN_RES, BETA, NB, BSEED)
check(f"tái lập lượt chính: mảng hạng (b) trùng tuyệt đối, mọi chỉ số và cặp của metrics_best.json ({REP['n']} phép so)", REP['ok'], str(REP['failed'][:3]))
R_MAIN = REP['R']
print(f'[tái lập] {time.time() - t0:.0f} s | SMOKE: {SMOKE_VIDS}' if SMOKE else f'[tái lập] {time.time() - t0:.0f} s')

In [ ]:
# ===== 3. Kiểm hash protocol bổ sung =====
PB_PATH = f'{FUSHION}/configs/htstep_uemr_bo_sung_protocol.json'
PH = sha256(PB_PATH)
check('sha256 protocol bổ sung', PH[:16] == PROTO_SHA16, f'tại {FUSHION_REF}: {PH[:16]}, notebook ghi {PROTO_SHA16}')
print('sha256 protocol bổ sung:', PH)
PB_COMMIT = git('log', '-1', '--format=%h %ad', '--date=short', '--', 'configs/htstep_uemr_bo_sung_protocol.json')
check('protocol bổ sung đã commit', bool(PB_COMMIT), PB_COMMIT)
PB = json.load(open(PB_PATH, encoding='utf-8'))
check('protocol bổ sung trỏ đúng lượt chính', PB['main']['protocol_sha16'] == MAIN_SHA16 and PB['main']['seg_hash'] == MAIN_SEG_HASH)
RES = f'{HB}/results/{PROTO_SHA16[:8]}'
os.makedirs(RES, exist_ok=True)
MAN_PATH = f'{HB}/run_manifest.json'
MAN = rj(MAN_PATH, {'runs': []})
RUN = {'started': time.strftime('%Y-%m-%d %H:%M:%S'), 'commit': COMMIT, 'protocol_sha16': PH[:16], 'flags': FLAGS, 'env': ENV, 'sec': {}, 'info': {}}
def done(part, t0, **info):
    RUN['sec'][part] = round(time.time() - t0, 1)
    RUN['info'][part] = info
    wj(MAN_PATH, {**MAN, 'runs': MAN['runs'] + [RUN]})
    print(f'[manifest] {part}: {RUN["sec"][part]} s')
done('main_reproduction', T_START, checks=REP['n'])
EVB = PB['eval']
LBL = {**hp.ROWS, **PB['labels']}

In [ ]:
# ===== 4. Bộ đoạn mới (đã sinh ở local, hash trong protocol), đếm clip mới, ước giờ =====
from omni_retrieval.manifest import segment_table, unique_segments, segment_records, caption_records, store_keys, read_store, write_jsonl, seg_key
t0 = time.time()
SEGB = f"{FUSHION}/{PB['segments']['dir']}"
NEW = {}
for name, s_ in PB['segments']['files'].items():
    p_ = f'{SEGB}/{name}.json'
    check(f'sha256 {name}.json = protocol', sha256(p_) == s_)
    NEW[name] = json.load(open(p_, encoding='utf-8'))
MSRC = rj(f'{MAIN_RES}/segments_SOURCE.json')
MAINSEG = {}
for n, f in hp.FILES.items():      # đoạn của lượt chính (ô 8 lượt chính ghi lên Drive), hash như SOURCE
    p_ = f'{HM}/segments/{f}.json'
    check(f'đoạn lượt chính {f}.json = SOURCE', sha256(p_) == MSRC['files'][f])
    MAINSEG[n] = json.load(open(p_, encoding='utf-8'))
R6P = MAINSEG['uniav_pred']
GT = hd.gt_partition(CAPS)
kp = {v: len(R6P[v]) for v in KHO}
for name in ('r4_s0', 'r4_s1', 'r4_s2', 'r5'):
    check(f'{name}: K = K_pred mọi video', all(len(NEW[name][v]) == kp[v] for v in KHO))
for name in ('r3kgt', 'r7n90', 'r7n70', 'r7n50', 'r7n30'):
    check(f'{name}: K = K_GT mọi video', all(len(NEW[name][v]) == len(GT[v]) for v in KHO))
check('r4: mọi đoạn >= d_min, phủ kín video', all(min(b - a for a, b in NEW[n][v]) >= PB['segments']['params']['d_min_r4'] - 0.011 and NEW[n][v][0][0] == 0.0
                                                   and abs(NEW[n][v][-1][1] - DUR[v]) < 0.011 for n in ('r4_s0', 'r4_s1', 'r4_s2') for v in KHO))
ROWS_NEW = ['r3kgt', 'r4_s0', 'r4_s1', 'r4_s2', 'r5', 'r7n90', 'r7n70', 'r7n50', 'r7n30'] + (['r6k12', 'r6k33'] if RUN_R6K else []) + (['r7fill'] if RUN_R7FILL else [])
SEG_ROWS = segment_table({n: {v: NEW[n][v] for v in VIDS} for n in ROWS_NEW})
UNIQ = unique_segments(SEG_ROWS)
AV = f"{OM('best')}/clips"
todo_av = segment_records(UNIQ, store_keys(AV, 'av'))
cnt = pd.DataFrame([{'dòng': LBL[n], 'đoạn': sum(len(NEW[n][v]) for v in VIDS), 'vec/video': np.mean([len(NEW[n][v]) for v in VIDS])} for n in ROWS_NEW])
display(cnt.round(2))
SEC_REF = 0.32
print(f'clip khác nhau: {len(UNIQ)} | chưa có trong store best: {len(todo_av)} | ước lượng A100: {len(todo_av) * SEC_REF / 3600:.2f} giờ')
done('segments', t0, unique=len(UNIQ), todo=len(todo_av))

In [ ]:
# ===== 5. mp4 val (như ô 5 lượt chính) và Omni best encode clip mới (cache cũ, fp16, failed.csv, > 2 % lỗi thì dừng) =====
import importlib.util as _ilu
from omni_retrieval.collect import collect_segments, collect_text
from omni_retrieval.manifest import PROMPT
_spec = _ilu.spec_from_file_location('htstep_io', f'{OMNI_FIX}/src/omniretriever/data/htstep_io.py')
hio = _ilu.module_from_spec(_spec); _spec.loader.exec_module(hio)
t0 = time.time()
items = hd.read_jsonl(items_p)
SRC = {r['video_id']: r['source'] for r in items}
TAR = {r['video_id']: (int(r['offset_data']), int(r['size'])) for r in csv.DictReader(open(f'{OMNI_FIX}/data/htstep/tar_index.csv', newline='', encoding='utf-8'))}
check('sha256 tar_index.csv', sha256(f'{OMNI_FIX}/data/htstep/tar_index.csv') == D['tar_index']['sha256'])
def fetch_one(v):
    dst = f'{VID}/{v}.mp4'
    if os.path.exists(dst) and os.path.getsize(dst):
        return v, 'có sẵn'
    tmp = f'/content/tmp_vid/{v}'
    os.makedirs(tmp, exist_ok=True)
    try:
        s, raw = SRC[v], f'{tmp}/raw.mp4'
        if 'tar' in s:
            hio.range_get(s['tar'], *TAR[v], raw, repo=s['repo'])
        else:
            hio.hf_get(s['repo'], s['file'], raw)
        shutil.copy2(raw, dst + '.part'); os.replace(dst + '.part', dst)
        return v, 'tải'
    except Exception as e:     # noqa: BLE001
        return v, f'lỗi: {type(e).__name__}: {e}'
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
def to_local(v):
    src, dst = f'{VID}/{v}.mp4', f'/content/videos/{v}.mp4'
    if os.path.exists(dst) and os.path.getsize(dst):
        return v, 'có sẵn'
    p_ = hio.probe(src)
    if p_['codec'] != 'av1':
        shutil.copy2(src, dst + '.part'); os.replace(dst + '.part', dst)
        return v, 'copy'
    ok, why, _ = hio.transcode(src, dst + '.h264.mp4', p_)
    if not ok:
        return v, f'lỗi transcode: {why}'
    os.replace(dst + '.h264.mp4', dst)
    return v, 'av1 -> h264'
NEED_MP4 = bool(todo_av) or RUN_TEACHER or RUN_PT
if NEED_MP4:
    free = shutil.disk_usage('/content').free / 2**30
    check('đĩa /content đủ cho video + bản H.264 của AV1 (>= 60 GB trống)', free >= 60, f'{free:.0f} GB')
    with ThreadPoolExecutor(4) as ex:
        STATUS = dict(ex.map(fetch_one, VIDS))
    bad = {v: s for v, s in STATUS.items() if s.startswith('lỗi')}
    check('mọi video mp4 có đủ', not bad, str(list(bad.items())[:3]))
    os.makedirs('/content/videos', exist_ok=True)
    with ThreadPoolExecutor(8) as pool:
        TO_LOCAL = dict(pool.map(to_local, VIDS))
    bad = {v: s for v, s in TO_LOCAL.items() if s.startswith('lỗi')}
    check('mp4 local đọc được bằng decord (AV1 đã chuyển H.264)', not bad, str(list(bad.items())[:3]))

BASE_MODEL, BEATS = '/content/WAVE_HOME/WAVE-7B', '/content/WAVE_HOME/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt'
PT_ADAPTER, BEST_ADAPTER = '/content/adapters/omniretriever-7b', '/content/adapters/best'
ADAPTERS = {'best': BEST_ADAPTER, 'pt': PT_ADAPTER}
_M = PR['models']
def wave_ok():
    idx = glob.glob(f'{BASE_MODEL}/*.index.json')
    return os.path.exists(BEATS) and bool(idx) and all(os.path.exists(f'{BASE_MODEL}/{s}') for s in set(json.load(open(idx[0]))['weight_map'].values()))
def adapters():
    if not os.path.exists(f'{PT_ADAPTER}/adapter_model.safetensors'):
        sh(f"hf download {_M['omni_pt']['repo']} --include 'omniretriever-7b/*' --local-dir /content/adapters")
    check('sha256 adapter pt', sha256(f'{PT_ADAPTER}/adapter_model.safetensors') == _M['omni_pt']['sha256'])
    if not os.path.exists(f'{BEST_ADAPTER}/adapter_model.safetensors'):
        os.makedirs(BEST_ADAPTER, exist_ok=True)
        src = f'{RUN_DRIVE}/eval/adapter_best'
        for f in os.listdir(src):
            shutil.copy2(f'{src}/{f}', f'{BEST_ADAPTER}/{f}')
    check('sha256 adapter best', sha256(f'{BEST_ADAPTER}/adapter_model.safetensors') == _M['omni_best']['sha256'])
def omni_env(tag):
    check('GPU đủ cho Omni 7B (>= 22 GB)', GPU_GB >= 22, f'{gpu}, {GPU_GB:.0f} GB')
    chk = subprocess.run([sys.executable, '-c', 'import decord, av, peft, torch, transformers; print(transformers.__version__, torch.cuda.is_available())'], capture_output=True, text=True)
    last = (chk.stdout.strip().splitlines() or [''])[-1]
    check('môi trường encode (transformers 4.51.3, CUDA)', chk.returncode == 0 and last.split()[:2] == ['4.51.3', 'True'], f'{last} {chk.stderr[-300:]}')
    if not wave_ok():
        sh('hf download nguyenminh04/omni-model --local-dir /content/WAVE_HOME --max-workers 2')
    check('WAVE-7B + BEATs đủ file', wave_ok())
    adapters()
    return f'cd {FUSHION} && python -m omni_retrieval.encode --base-model {BASE_MODEL} --beats-path {BEATS} --adapter {ADAPTERS[tag]} --video-root /content/videos'
def encode(modality, manifest, store, log, enc, extra=''):
    with open(log, 'a') as f:
        f.write(f"\n===== {modality} | {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n")
    get_ipython().system(f'{enc} --modality {modality} --manifest "{manifest}" --store "{store}" {extra} 2>&1 | tee -a "{log}"')
    ids = [json.loads(l)['id'] for l in open(manifest) if l.strip()]
    return sorted(set(ids) - store_keys(store, modality)), len(ids)
def last_speed(store):
    runs = [json.loads(l) for l in open(f'{store}/runs.jsonl') if l.strip()] if os.path.exists(f'{store}/runs.jsonl') else []
    return runs[-1] if runs else {}

ENC_INFO = {}
caps_rec = [{'caption_id': c['caption_id'], 'sentence': c['sentence']} for c in CAPS]
jobs = [('best', UNIQ)]
if RUN_PT:        # phần F: Omni pt trên R0, R3, R6, R8, R7 của lượt chính
    PT_ROWS = ['global', 'uni_M', 'uniav_pred', 'r8', 'gt']
    PT_SEG_ROWS = segment_table({n: {v: MAINSEG[n][v] for v in VIDS} for n in PT_ROWS})
    jobs.append(('pt', unique_segments(PT_SEG_ROWS)))
for tag, uniq in jobs:
    store = f'{OM(tag)}/clips'
    os.makedirs(OM(tag), exist_ok=True)
    left = segment_records(uniq, store_keys(store, 'av'))
    if left:
        enc = omni_env(tag)
        write_jsonl(f'{HB}/segs_todo_{tag}.jsonl', left)
        encode('av', f'{HB}/segs_todo_{tag}.jsonl', store, f'{HB}/encode_{tag}.log', enc, '--num-workers 4')
        ENC_INFO[tag] = last_speed(store)
    miss = [r['id'] for r in segment_records(uniq, store_keys(store, 'av'))]
    pd.DataFrame({'seg_key': miss}).to_csv(f'{HB}/failed_{tag}.csv', index=False)
    check(f'{tag}: clip lỗi <= 2%', 100 * len(miss) / max(len(uniq), 1) <= 2, f'{len(miss)} / {len(uniq)} clip, xem {HB}/failed_{tag}.csv')
    if tag == 'pt':
        tx = f"{OM('pt')}/text"
        tx_todo = caption_records(caps_rec, store_keys(tx, 'text'))
        if tx_todo:
            write_jsonl(f'{HB}/caps_todo_pt.jsonl', tx_todo)
            left_t, _ = encode('text', f'{HB}/caps_todo_pt.jsonl', tx, f'{HB}/encode_pt.log', omni_env('pt'), '--batch-size 32')
            check('pt: encode đủ câu hỏi', not left_t, str(left_t[:5]))
DBB = f'{LOCAL}/db_best'
os.makedirs(DBB, exist_ok=True)
SEGB_INFO = collect_segments(SEG_ROWS, AV, f'{DBB}/segments.npz', allow_missing=True)
os.makedirs(f'{HB}/db_best', exist_ok=True); shutil.copy2(f'{DBB}/segments.npz', f'{HB}/db_best/segments.npz')
if RUN_PT:
    DBP = f'{LOCAL}/db_pt'; os.makedirs(DBP, exist_ok=True)
    collect_segments(PT_SEG_ROWS, f"{OM('pt')}/clips", f'{DBP}/segments.npz', allow_missing=True)
    collect_text([{k: c[k] for k in ('caption_id', 'video_id', 'sentence', 'gt_ts', 'gt_te', 'gt_seg_key')} for c in CAPS], f"{OM('pt')}/text", f'{DBP}/text.npz')
print('[best] segments mới:', SEGB_INFO, '| tốc độ:', {k: v.get('sec_per_clip') for k, v in ENC_INFO.items()})
done('encode', t0, info={k: {x: v.get(x) for x in ('encoded', 'failed', 'sec_per_clip', 'mean_sec_per_clip')} for k, v in ENC_INFO.items()})
torch.cuda.empty_cache()

In [ ]:
# ===== 6. (ưu tiên 2) Môi trường thầy: kiểm adapter v4, encode_av đoạn R6 (mục 5.5) =====
t0 = time.time()
TR6 = None
if RUN_TEACHER:
    TEACHER_ADAPTER = '/content/adapters/teacher'
    T = _M['omni_teacher']
    adapters()
    teacher_lora = hf_hub_download(T['repo'], f"{T['dir']}/{T['file']}", local_dir=f'{A}/teacher')
    check('sha256 LoRA thầy', sha256(teacher_lora) == T['sha256'])
    if not os.path.exists(f'{TEACHER_ADAPTER}/adapter_model.safetensors'):
        sh(f'cd {OMNI_FIX} && PYTHONPATH={OMNI_FIX}/src python scripts/materialize_adapter.py {PT_ADAPTER} {os.path.dirname(teacher_lora)} {TEACHER_ADAPTER}')
    check('sha256 adapter thầy (LoRA + head pt) = protocol', sha256(f'{TEACHER_ADAPTER}/adapter_model.safetensors') == T['adapter_sha256'])
    VENV = '/content/venv_teacher'
    if not os.path.exists(f'{VENV}/bin/python'):
        sh(f'{sys.executable} -m venv --system-site-packages --without-pip {VENV}')
        sh(f'{sys.executable} -m pip --python {VENV}/bin/python install -q transformers==4.57.3 peft==0.20.0 av')
    v_ = subprocess.run([f'{VENV}/bin/python', '-c', 'import transformers, peft, av, torch; print(transformers.__version__, peft.__version__, torch.cuda.is_available())'], capture_output=True, text=True)
    check('môi trường thầy (transformers 4.57.3, peft 0.20.0, CUDA)', v_.stdout.split()[:3] == ['4.57.3', '0.20.0', 'True'], v_.stdout + v_.stderr[-300:])
    if not wave_ok():
        sh('hf download nguyenminh04/omni-model --local-dir /content/WAVE_HOME --max-workers 2')
    TC = PR['reproduction']['teacher_clip_check']
    SAMPLE = TC['sample']
    EXTRA = tuple(SAMPLE['extra'])
    _v = set(VIDS)
    pool = sorted(k for k in TEACHER if k[0] in _v and k != EXTRA)
    chosen = (pool[:SAMPLE['n']] if SMOKE else [tuple(k) for k in SAMPLE['clips']]) + [EXTRA]
    TD = f'{HB}/teacher_r6'
    os.makedirs(TD, exist_ok=True)
    wj(f'{TD}/clips.json', [{'id': '|'.join(map(str, k)), 'video': f'{VID}/{k[0]}.mp4', 'start': TEACHER[k]['segment'][0], 'end': TEACHER[k]['segment'][1],
                             'npz': TEACHER[k]['npz'], 'row': TEACHER[k]['row']} for k in chosen])
    write_jsonl(f'{TD}/caps.jsonl', caption_records(caps_rec, set()))
    r6_clips = sorted({seg_key(v, a, b): {'id': seg_key(v, a, b), 'video': f'{VID}/{v}.mp4', 'start': a, 'end': b} for v in VIDS for a, b in R6P[v]}.values(), key=lambda c: c['id'])
    write_jsonl(f'{TD}/r6_clips.jsonl', r6_clips)
    if os.path.exists(f'{TD}/report.json'):
        os.remove(f'{TD}/report.json')
    r = subprocess.run([f'{VENV}/bin/python', '-m', 'uemr_htstep.teacher_encode', '--base-model', BASE_MODEL, '--adapter', TEACHER_ADAPTER, '--clips', f'{TD}/clips.json',
                        '--captions', f'{TD}/caps.jsonl', '--store', f"{OM('student_text')}/text", '--report', f'{TD}/report.json', '--best-adapter', BEST_ADAPTER,
                        '--repeat-id', '|'.join(map(str, EXTRA)), '--repeats', str(TC['stability']['extra_runs']), '--min-mean', str(TC['absolute']['mean_min']),
                        '--min-clip', str(TC['absolute']['clip_min']), '--ratio', str(TC['discrimination']['mean_ratio_min']), '--list-below', str(TC['list_below']),
                        '--av-clips', f'{TD}/r6_clips.jsonl', '--av-store', f'{TD}/av'],
                       cwd=FUSHION, capture_output=True, text=True, env={**os.environ, 'PYTHONPATH': f'{OMNI_AV}/src:{FUSHION}', 'WAVE_HOME': '/content/WAVE_HOME', 'BEATS_PATH': BEATS})
    print(r.stdout[-600:]); print(r.stderr[-800:] if r.returncode else '')
    REP_T = rj(f'{TD}/report.json', {})
    check('kiểm adapter thầy v4 (phân biệt + chặn tuyệt đối)', REP_T.get('ok') is True, f"mean {REP_T.get('mean')}, best {REP_T.get('best_mean')}, tỉ số {REP_T.get('gap_ratio')}")
    check('teacher_encode chạy hết', r.returncode == 0, r.stderr[-300:])
    tav = read_store(f'{TD}/av', 'av')
    miss = [c['id'] for c in r6_clips if c['id'] not in tav]
    check('thầy: clip R6 lỗi <= 2%', 100 * len(miss) / len(r6_clips) <= 2, f'{len(miss)} / {len(r6_clips)}')
    # vector trung bình của teacher train (OmniCosC như model card): mọi đoạn GT của video train UniAV
    ids_tr = json.load(open(f"{FUSHION}/{D['train_ids']['file']}", encoding='utf-8'))['uniav_train']
    TRD = snapshot_download(PB['teacher']['train_repo'], repo_type='dataset', allow_patterns=[f'{PB["teacher"]["train_dir"]}/*.npz'], local_dir=f'{A}/teacher_train')
    tot, n_ = np.zeros(3584), 0
    for v in ids_tr:
        p_ = f"{TRD}/{PB['teacher']['train_dir']}/{v}.npz"
        if os.path.exists(p_):
            with np.load(p_) as z:
                tot += z['emb'].astype(np.float64).sum(0); n_ += len(z['emb'])
    TMEAN = (tot / max(n_, 1)).astype(np.float32)
    def omnicos(s, t, mean):
        u = lambda x: x / np.linalg.norm(x, axis=1, keepdims=True)
        return 100 * float((u(s) * u(t)).sum(1).mean()), 100 * float((u(s - mean) * u(t - mean)).sum(1).mean())
    # kiểm: trò (article GT) trên đoạn GT có vector thầy phải cho lại số model card
    with np.load(f'{BIN}/student_r7_gt.npz') as z:
        stu_gt = {hm.row_key(v, a, b): e.astype(np.float32) for v, a, b, e in zip(z['video_id'], z['ts'], z['te'], z['emb'])}
    keep_q, by_clip = hd.teacher_queries(CAPS, VAL_ROWS, TEACHER)
    kk = sorted({CAPS[i]['gt_seg_key']: CAPS[i] for i in keep_q}.items())
    s_, t_ = np.stack([stu_gt[hm.row_key(c['video_id'], c['gt_ts'], c['gt_te'])] for _, c in kk]), np.stack([by_clip[k] for k, _ in kk])
    cos_gt = omnicos(s_, t_, TMEAN)
    MC = PB['teacher']['model_card']
    print(f'đoạn GT, article GT: OmniCos {cos_gt[0]:.2f} (model card {MC["OmniCos"]}), OmniCosC {cos_gt[1]:.2f} (model card {MC["OmniCosC"]}), {len(kk)} đoạn, train mean từ {n_} vector')
    check('OmniCos / OmniCosC tái lập model card (vector trung bình train đúng)', abs(cos_gt[0] - MC['OmniCos']) <= MC['tol'] and abs(cos_gt[1] - MC['OmniCosC']) <= MC['tol'], str(cos_gt))
    i6 = MAIN.index('uniav_pred')
    ok6 = np.array([seg_key(v, a, b) in tav for v, a, b in zip(i6.seg_video, i6.ts, i6.te)])
    e_t6 = np.stack([tav[seg_key(v, a, b)] if k else np.zeros(3584, np.float32) for v, a, b, k in zip(i6.seg_video, i6.ts, i6.te, ok6)])
    e_s6 = MAIN.student(i6, 'r6')
    cos_r6 = omnicos(e_s6[ok6], e_t6[ok6], TMEAN)
    TR6 = {'segments': int(ok6.sum()), 'OmniCos R6': cos_r6[0], 'OmniCosC R6': cos_r6[1], 'OmniCos GT (article GT)': cos_gt[0], 'OmniCosC GT (article GT)': cos_gt[1],
           'teacher_check': {k: REP_T.get(k) for k in ('n', 'mean', 'min', 'best_mean', 'gap_ratio', 'below', 'stability')}, 'av_clips': REP_T.get('av_clips', {}).get('sec_per_clip'),
           'e_t6': e_t6 / np.clip(np.linalg.norm(e_t6, axis=1, keepdims=True), 1e-12, None), 'ok6': ok6}
    print({k: v for k, v in TR6.items() if k not in ('e_t6', 'ok6')})
done('teacher', t0, **({k: v for k, v in TR6.items() if k not in ('e_t6', 'ok6')} if TR6 else {'skipped': True}))

In [ ]:
# ===== 7. Vector student (S)_pred / (S)_gt / (S)_rand trên R8 (đã tính ở local; kiểm hash và trùng cache) =====
t0 = time.time()
for f, s_ in PB['student_extra'].items():
    check(f'sha256 {f} (bo_sung_in) = protocol', os.path.exists(f'{BIN}/{f}') and sha256(f'{BIN}/{f}') == s_, 'thiếu file: tải D:/Học/KL/output/htstep_uemr/bo_sung/uniav/' + f + f' lên {BIN}' if not os.path.exists(f'{BIN}/{f}') else '')
i8 = MAIN.index('r8')
with np.load(f'{BIN}/student_r8.npz') as z:
    e_pred_new = hm.student_rows(i8, z['video_id'], z['ts'], z['te'], z['emb'])
e_pred = MAIN.student(i8, 'r8')
u = lambda x: x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)
cmin = float((u(e_pred_new) * u(e_pred)).sum(1).min())
check('(S)_pred tính lại = cache lượt chính (cos >= 0.9999)', cmin >= PB['student_check_cos'], f'cos min {cmin:.6f}')
E_S8 = {'pred': e_pred}
for how in ('gt', 'rand'):
    with np.load(f'{BIN}/student_r8_{how}.npz') as z:
        E_S8[how] = hm.student_rows(i8, z['video_id'], z['ts'], z['te'], z['emb'])
        if how == 'rand':
            with np.load(f'{BIN}/student_r8.npz') as zp:
                top1 = dict(zip(zp['article_video'], zp['article']))
            check('(S)_rand: article ngẫu nhiên khác article dự đoán ở mọi video', all(a_ != top1[v] for v, a_ in zip(z['article_video'], z['article'])))
done('student_extra', t0, cos_min=cmin)

In [ ]:
# ===== 8. Context Fusion cross-fitting (mục 5.2): fold theo article, 4 dòng + Z xáo =====
from uemr_htstep import cf
t0 = time.time()
CF = PB['context_fusion']
HPF = dict(CF['hyper'], epochs=1) if SMOKE else CF['hyper']
check('sha256 feats.npz (bo_sung_in) = protocol', sha256(f'{BIN}/feats.npz') == PB['feats_sha256'])
shutil.copy2(f'{BIN}/feats.npz', f'{LOCAL}/feats.npz')
with np.load(f'{LOCAL}/feats.npz') as z:
    FEATS = {str(v): (f_, t_) for v, f_, t_ in zip(z['video_id'], z['feat'], z['t_sec'])}
FOLDS = {**{v: 'A' for v in PB['folds']['A']}, **{v: 'B' for v in PB['folds']['B']}}
check('fold: đủ kho, không article nào ở hai fold', set(FOLDS) == set(KHO))
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
CFX, CF_REP = {}, {}
for name, part, shuffle in (('R0+CF', 'global', False), ('R3+CF', 'uni_M', False), ('R6+CF', 'uniav_pred', False), ('R8+CF', 'r8', False), ('R8+CF, Z xáo', 'r8', True)):
    out = f'{HB}/cf/{name.replace(" ", "").replace(",", "_").replace("+", "_")}.npz'
    if os.path.exists(out) and not SMOKE:
        with np.load(out) as z:
            from omni_retrieval.search import SegmentIndex
            CFX[name] = SegmentIndex(z['emb'], z['video_id'], z['ts'], z['te'], name)
        CF_REP[name] = rj(out.replace('.npz', '.json'))
        print(f'[cf] {name}: có sẵn'); continue
    idx, rep = cf.cross_fit(MAIN.index(part), MAIN.index('global'), FEATS, DUR, CAPS, MAIN.text['emb'], FOLDS, HPF, SEED, DEV, shuffle,
                            log=(lambda *_: None) if SMOKE else print)
    CFX[name], CF_REP[name] = idx, rep
    os.makedirs(os.path.dirname(out), exist_ok=True)
    np.savez(out, emb=idx.emb.astype(np.float16), video_id=idx.seg_video, ts=idx.ts, te=idx.te)
    wj(out.replace('.npz', '.json'), rep)
    alphas = ', '.join(f"{k}: {v['history'][-1]['alpha']:+.4f}" for k, v in rep.items())
    print(f'[cf] {name}: alpha {alphas} | {idx.emb.shape}')
torch.cuda.empty_cache()
done('context_fusion', t0, alpha={n: {k: v['history'][-1]['alpha'] for k, v in r_.items()} for n, r_ in CF_REP.items()})

In [ ]:
# ===== 9. Chấm mọi dòng / cột mới, strict + multi-positive, lưu thứ hạng =====
from omni_retrieval.evaluate import rank_metrics
from omni_retrieval.search import SegmentIndex
t0 = time.time()
TXT, TXS, VIDQ = MAIN.text, MAIN.text_s, MAIN.vids
NEWX = lambda n: SegmentIndex.load(DBB, n)
RB = {}                       # tên -> kết quả run (mode -> ranks, vcmr, top_row)
IDX = {}
def put(name, index, **kw):
    IDX[name] = index
    RB[name] = sc.run(index, TXT, CAPS, **kw)
for n in ('r3kgt', 'r4_s0', 'r4_s1', 'r4_s2', 'r5', 'r7n90', 'r7n70', 'r7n50', 'r7n30') + (('r6k12', 'r6k33') if RUN_R6K else ()) + (('r7fill',) if RUN_R7FILL else ()):
    put(LBL[n], NEWX(n))
put('R1', MAIN.index('uniav_pred').mean_pooled('r1'))
# C1z, (U)z: thống kê trên mọi cặp (câu val, đoạn R8), không nhãn
i8 = MAIN.index('r8')
sim_b8, sim_s8 = TXT['emb'] @ i8.emb.T, TXS['emb'] @ E_S8['pred'].T
ZB, ZS = bs.zstats(sim_b8), bs.zstats(sim_s8)
fill8 = MAIN.fill_mask(i8)
put('C1z', i8, sim=bs.c1z(sim_b8, sim_s8, fill8, ZB, ZS))
put('(U)z', i8, sim=(sim_b8 - ZB[0]) / ZB[1] + (sim_s8 - ZS[0]) / ZS[1])
for how in ('pred', 'gt', 'rand'):
    put(f'(S)_{how}', i8, sim=sim_s8 if how == 'pred' else TXS['emb'] @ E_S8[how].T)
del sim_b8, sim_s8
G0 = MAIN.index('global')
for p_ in ('uni_M', 'uniav_pred', 'r8'):
    ix = MAIN.index(p_)
    put(f'{hp.ROWS[p_]}+G', ix, bonus=bs.global_bonus(TXT['emb'], ix, G0, PB['columns']['G']['gamma']))
for n, ix in CFX.items():
    put(n, ix)
# Exp 5 (mô tả): cách gộp điểm trên R6, R8 (b)
AG = PB['aggregation']
RAWU = {r['video_id']: r for r in map(json.loads, filter(str.strip, open(f'{UNIAV_DIR}/uniav_raw.jsonl', encoding='utf-8')))}
CONF = bs.r6_conf({v: RAWU[v]['pred'] for v in KHO}, DUR, PR['resolved']['theta'], PR['params']['r6_postprocess'])
check('R6: mọi đoạn có conf', all(CONF.get(hm.row_key(v, a, b)) is not None for v in KHO for a, b in R6P[v]))
MED = {v: float(np.median([CONF[hm.row_key(v, a, b)] for a, b in R6P[v]])) for v in KHO}     # đoạn lấp: conf = trung vị conf R6 của video
for p_ in ('uniav_pred', 'r8'):
    ix = MAIN.index(p_)
    put(f'{hp.ROWS[p_]} top-2', ix, agg='topk_mean', k=AG['topk_k'])
    put(f'{hp.ROWS[p_]} LSE', ix, agg='lse', tau=AG['lse_tau'])
    conf = np.array([CONF.get(hm.row_key(v, a, b), MED[v]) for v, a, b in zip(ix.seg_video, ix.ts, ix.te)], np.float64)
    put(f'{hp.ROWS[p_]} max+log conf', SegmentIndex(ix.emb, ix.seg_video, ix.ts, ix.te, 'conf', bias=AG['conf_beta'] * np.log(np.clip(conf, 1e-6, None))))
if TR6 is not None:     # (T): text thầy · AV thầy trên đoạn R6
    i6 = MAIN.index('uniav_pred')
    put('(T) R6', i6, sim=np.where(TR6['ok6'][None, :], TXS['emb'] @ TR6['e_t6'].T, -np.inf))
    put('(S) R6', i6, sim=TXS['emb'] @ MAIN.student(i6, 'r6').T)
if RUN_PT:
    from omni_retrieval.evaluate import load_text
    TPT = load_text(DBP)
    for p_ in PT_ROWS:
        RB[f'pt {hp.ROWS[p_]}'] = sc.run(SegmentIndex.load(DBP, p_), TPT, CAPS)
        IDX[f'pt {hp.ROWS[p_]}'] = SegmentIndex.load(DBP, p_)
TAB = []
for n, res in RB.items():
    pooled = n == 'R1'
    for m, met in sc.row_metrics(res, MAIN.full, IDX.get(n), pooled).items():
        TAB.append({'mode': m, 'dòng / cột': n, **met})
    if not SMOKE:
        wj(f'{RES}/ranks_{n.replace(" ", "_").replace("/", "_")}.json', {'caption_id': [c['caption_id'] for c in CAPS], **{m: res[m]['ranks'].tolist() for m in sc.MODES},
                                                                         **{f'vcmr_{m}': res[m]['vcmr'].astype(int).tolist() for m in sc.MODES}})
R4R = {m: np.stack([RB[LBL[f'r4_s{s}']][m]['ranks'] for s in (0, 1, 2)]) for m in sc.MODES}
R4V = {m: np.stack([sc.vcmr_rank(RB[LBL[f'r4_s{s}']][m]['vcmr']) for s in (0, 1, 2)]) for m in sc.MODES}
def ranks_of(name, m):
    if name == 'R4̄':
        return R4R[m], R4V[m]
    if name in RB:
        return RB[name][m]['ranks'], sc.vcmr_rank(RB[name][m]['vcmr'])
    col, p_ = name.split(':') if ':' in name else ('b', name)
    r_ = R_MAIN[(col, {v: k for k, v in hp.ROWS.items()}[p_])][m]
    return r_['ranks'], sc.vcmr_rank(r_['vcmr'])
PAIRS_B = PB['pairs']
PBT = []
for crit, a_, b_ in PAIRS_B:
    if any(x not in RB and x != 'R4̄' and x.split(':')[-1] not in hp.ROWS.values() for x in (a_, b_)):
        continue
    for m in sc.MODES:
        ra, va = ranks_of(a_, m); rb, vb = ranks_of(b_, m)
        PBT.append({'tiêu chí': crit, 'mode': m, 'chỉ số': 'R@1', 'cặp': f'{a_} - {b_}', **sc.boot(ra, rb, VIDQ, NB, BSEED)})
        if 'R1' not in (a_, b_) and 'R0' not in (a_, b_):
            PBT.append({'tiêu chí': crit, 'mode': m, 'chỉ số': 'VCMR R@1', 'cặp': f'{a_} - {b_}', **sc.boot(va, vb, VIDQ, NB, BSEED)})
tab, pbt = pd.DataFrame(TAB), pd.DataFrame(PBT)
R4SEED = {m: [100 * float((R4R[m][s] <= 1).mean()) for s in range(3)] for m in sc.MODES}
if SMOKE:
    print(f'[smoke] bảng {tab.shape}, cặp {pbt.shape}, R4 3 seed: {R4R["multi"].shape}')
else:
    print('=== multi-positive ==='); display(tab[tab['mode'] == 'multi'].drop(columns=['mode']).round(2))
    print('=== strict ==='); display(tab[tab['mode'] == 'strict'].drop(columns=['mode']).round(2))
    print('R4 R@1 từng seed:', R4SEED)
    display(pbt.drop(columns=['ci_low', 'ci_high']).round(2))
wj(f'{RES}/metrics_bo_sung.json', {'protocol_sha16': PH[:16], 'main_protocol_sha16': MAIN_SHA16, 'table': TAB, 'pairs': PBT, 'r4_seeds_R@1': R4SEED,
                                   'zstats': {'b': ZB, 'S': ZS}, 'teacher_r6': {k: v for k, v in (TR6 or {}).items() if k not in ('e_t6', 'ok6')},
                                   'cf': CF_REP, 'bootstrap': PR['eval']['bootstrap'], 'commit': COMMIT, 'smoke': SMOKE})
done('eval', t0, rows=len(RB))

In [ ]:
# ===== 10. Phân tích mục 6, đo tốc độ (mục 5.3) =====
t0 = time.time()
GRP = {}
missed = bs.missed_by(R6P, CAPS)
shared = bs.shared_text(CAPS)
def sub_boot(a, b, mask):
    return sc.boot(a[mask], b[mask], VIDQ[mask], NB, BSEED)
def r1v(name, m, mask):
    ra, va = ranks_of(name, m)
    return {'R@1': 100 * float((ra[mask] <= 1).mean()), 'VCMR R@1': 100 * float((va[mask] == 1).mean()), 'câu': int(mask.sum())}
for m in sc.MODES:
    for gname, mask in (('bước bị R6 bỏ sót', missed), ('bước R6 bắt được', ~missed)):
        GRP[f'{m} | {gname}'] = {'dòng': {n: r1v(n, m, mask) for n in ('R0', 'R3', 'R6', 'R8', 'R2')},
                                 'R6 - R3': sub_boot(ranks_of('R6', m)[0], ranks_of('R3', m)[0], mask), 'R8 - R6': sub_boot(ranks_of('R8', m)[0], ranks_of('R6', m)[0], mask)}
    for gname, mask in (('text ở 1 video', ~shared), ('text ở >= 2 video', shared)):
        GRP[f'{m} | {gname}'] = {'R8 - R0': sub_boot(ranks_of('R8', m)[0], ranks_of('R0', m)[0], mask), 'câu': int(mask.sum())}
FILLTOP = {}
for name, res in (('(b)', R_MAIN[('b', 'r8')]), ('(S)', R_MAIN[('S', 'r8')]), ('(U)', R_MAIN[('U', 'r8')]), ('C1', R_MAIN[('C1', 'r8')]), ('C1z', RB['C1z'])):
    for m in sc.MODES:
        top_fill = fill8[res[m]['top_row']]
        FILLTOP[f'{name} {m}'] = {'% câu có đoạn đầu là đoạn lấp': 100 * float(top_fill.mean()),
                                  'VCMR R@1 khi đoạn đầu là đoạn lấp': 100 * float(res[m]['vcmr'][top_fill].mean()) if top_fill.any() else None,
                                  'VCMR R@1 khi đoạn đầu là đoạn R6': 100 * float(res[m]['vcmr'][~top_fill].mean()) if (~top_fill).any() else None}
from scipy.stats import spearmanr
CORR = {m: {'spearman hạng video đúng (b) vs (S), R8': float(spearmanr(R_MAIN[('b', 'r8')][m]['ranks'], R_MAIN[('S', 'r8')][m]['ranks']).correlation),
            'hit@1': bs.hit_table(R_MAIN[('b', 'r8')][m]['ranks'], R_MAIN[('S', 'r8')][m]['ranks'])} for m in sc.MODES}
spc = (ENC_INFO.get('best') or {}).get('mean_sec_per_clip') or (ENC_INFO.get('best') or {}).get('sec_per_clip')
k8, k6 = float(np.mean(i8.counts)), float(np.mean(MAIN.index('uniav_pred').counts))
COST = {'Omni best s/clip (A100, lượt encode ô 5)': spc, 'clip / video R8': k8, 'clip / video R6 (= C1z)': k6,
        'R8 (b): giây Omni / video': spc * k8 if spc else None, 'C1z: giây Omni / video': spc * k6 if spc else None,
        'UniAV forward + span head': PB['cost']['uniav_note'], 'trích K400 + BEATs': PB['cost']['feature_note']}
if SMOKE:
    print('[smoke] nhóm', len(GRP), '| fill-top', len(FILLTOP), '| corr', list(CORR), '| cost', {k: v for k, v in COST.items() if not isinstance(v, str)})
else:
    print(json.dumps({'nhóm': GRP, 'đoạn đầu là đoạn lấp': FILLTOP, 'tương quan': CORR, 'chi phí': COST}, indent=1, ensure_ascii=False, default=float))
wj(f'{RES}/groups.json', {'groups': GRP, 'fill_top': FILLTOP, 'correlation': CORR})
wj(f'{RES}/cost.json', COST)
done('analysis', t0)

In [ ]:
# ===== 11. Tiêu chí mục 8 -> criteria_bo_sung.json =====
t0 = time.time()
def pair(cap, mode='multi', what='R@1'):
    return next((p for p in PBT if p['mode'] == mode and p['chỉ số'] == what and p['cặp'] == cap), None)
CRB = {'protocol_sha16': PH[:16], 'main_protocol_sha16': MAIN_SHA16, 'main_conclusion_unchanged': PB['main']['conclusion'], 'note': PB['note'], 'criteria': {}}
for k, c in PB['criteria'].items():
    if c.get('kind') == 'level':      # B1: mức tIoU nhỏ nhất có cận dưới > 0
        res = {lv: pair(f'{lv} - {c["vs"]}') for lv in c['levels']}
        ok = [lv for lv in c['levels'] if res[lv] and res[lv]['ci_low'] > 0]
        CRB['criteria'][k] = {'test': c['test'], 'kết quả': {lv: (r_ and {x: r_[x] for x in ('A - B', 'CI', 'ci_low')}) for lv, r_ in res.items()},
                              'mức nhỏ nhất đạt': ok[-1] if ok else None, 'câu': c['pass'].replace('X', PB['labels_tiou'][ok[-1]]) if ok else c['fail']}
        continue
    p_ = pair(c['pair'])
    if p_ is None:
        CRB['criteria'][k] = {'test': c['test'], 'kết quả': None, 'câu': 'không chạy (thiếu dòng / cột)'}
        continue
    passed = bool(p_['ci_low'] > 0) if c.get('kind') != 'report' else None
    CRB['criteria'][k] = {'test': c['test'], 'kết quả': {x: p_[x] for x in ('A - B', 'CI', 'ci_low')}, 'đạt': passed,
                          'câu': c.get('pass') if passed else c.get('fail') if passed is False else 'chỉ báo'}
a4 = CRB['criteria'].get('A4', {})
if a4.get('đạt') is False:
    for k in ('A1', 'A2', 'B1'):
        if k in CRB['criteria']:
            CRB['criteria'][k]['kèm câu'] = PB['a4_caveat']
if SMOKE:
    print('[smoke] criteria:', sorted(CRB['criteria']))
else:
    print(json.dumps(CRB, indent=1, ensure_ascii=False))
wj(f'{RES}/criteria_bo_sung.json', CRB)
done('criteria', t0)

In [ ]:
# ===== 12. Xuất kết quả, tóm tắt, sha256 =====
import zipfile
t0 = time.time()
shutil.copy2(f'{SEGB}/prepared.json', f'{RES}/segments_SOURCE.json')
wj(f'{RES}/cf_history.json', {n: {k: v['history'] for k, v in r_.items()} for n, r_ in CF_REP.items()})
if SMOKE:
    check('smoke: đủ file kết quả', all(os.path.exists(f'{RES}/{f}') for f in ('metrics_bo_sung.json', 'groups.json', 'cost.json', 'criteria_bo_sung.json')))
    done('smoke', t0, videos=len(VIDS))
    shutil.rmtree(HB); shutil.rmtree(LOCAL, ignore_errors=True)
    print('SMOKE XONG: đã xoá bo_sung/smoke. Đổi SMOKE = False ở ô 0 rồi Run all.')
else:
    files = sorted(glob.glob(f'{RES}/*.json'))
    ZIP = f'/content/uemr_htstep_bo_sung_{PROTO_SHA16[:8]}.zip'
    with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in files + [MAN_PATH]:
            z.write(f, os.path.relpath(f, HB))
    for f in files:
        print(f'  {os.path.basename(f):40s} {os.path.getsize(f):>10d} B  sha256 {sha256(f)[:16]}')
    print('zip (chép vào Fushion results/uemr_htstep_bo_sung/%s/):' % PROTO_SHA16[:8], ZIP)
    done('export', t0, minutes=round((time.time() - T_START) / 60, 1))
    try:
        from google.colab import files as _f
        _f.download(ZIP)
    except Exception as e:     # noqa: BLE001
        print('tải zip thủ công:', e)